# Credresolve Collections Analytics — Master Analysis Notebook

**Executive Summary & Reproducible Technical Investigation**
- **Dataset Scope**: Jan 1, 2026 – Jul 31, 2026 (7 complete calendar months) | August 2026 (Partial 8-day monitoring dataset excluded from main trend conclusions).
- **Golden Dataset (Option A)**: 639,185 Raw Business Records → 585,963 Golden Records across 17 source tables (`golden_payments.csv` = 22,813 rows).
- **Payment Duplication**: 3,745 duplicate payment references (2,033 SUCCESS payment references had multiple raw records; 2,187 SUCCESS rows removed), eliminating ₹191.91 Cr (14.31%) of inflated recovery figure.
- **11% MoM Improvement Claim**: **NOT SUPPORTED**. The February-to-March raw recovery increase of approximately 10.99% coincides with a substantial increase in payment duplication; after Golden payment deduplication, the independent recovery metrics do not support the reported 11% improvement.
- **Recovery Trend**: The Golden recovery rate deteriorated substantially over the seven-month period, falling from 40.07% in January to 32.55% in July, despite a small rebound in June.
- **Simpson's Paradox**: **NOT PRESENT**. Performance dropped within every single DPD bucket (mix-adjusted decline: -5.34%).
- **Campaign Strategy Lift**: **-0.06 pp (Correlation)**. 95% CI [-1.40 pp, +1.27 pp] includes zero; statistically indistinguishable from zero.
- **Single Recommendation**: **Option 2 — More Collection Agents (₹10 Cr)** | 238 New Agents | Base Annualized Inc Recovery: ₹29.50 Cr | ROI: **2.95x** | Break-even: **~4.1 months** (Confidence: **LOW**).

In [1]:
import os, sys, json
import pandas as pd
import numpy as np
from config import DATA_DIR, OUTPUT_DIR, GOLDEN_DIR, DATASETS

raw = {name: pd.read_csv(os.path.join(DATA_DIR, f'{name}.csv'), low_memory=False) for name in DATASETS if os.path.exists(os.path.join(DATA_DIR, f'{name}.csv'))}
total_raw = sum(len(df) for df in raw.values())
print('=== PHASE 1: SYSTEM ENVIRONMENT & DATA LOAD ===')
print(f'Loaded 17 Raw Business Tables (Total Raw Rows: {total_raw:,})')
print(f"Raw Payments Rows: {len(raw['payments']):,} | Raw Agent Rows: {len(raw['agents']):,} | Raw Calls Rows: {len(raw['calls']):,}")

=== PHASE 1: SYSTEM ENVIRONMENT & DATA LOAD ===
Loaded 17 Raw Business Tables (Total Raw Rows: 639,185)
Raw Payments Rows: 25,500 | Raw Agent Rows: 30,000 | Raw Calls Rows: 91,350


## Phase 2 & 3: Data Forensics & Option A Golden Dataset (17 Tables)
Reconciling 639,185 raw business rows - 2,957 rejected exact dupes - 50,265 corrected PK/ref dupes = 585,963 Golden Rows across 17 tables.

In [2]:
pay = raw['payments'].copy()
raw_succ = pay[pay['payment_status'] == 'SUCCESS']
raw_succ_amt = raw_succ['amount'].sum()
succ_pk = raw_succ.drop_duplicates(subset=['payment_id'])
succ_dedup = succ_pk.sort_values('event_at').groupby('payment_reference').first().reset_index()
golden_succ_amt = succ_dedup['amount'].sum()
inflation_amt = raw_succ_amt - golden_succ_amt
inflation_pct = (inflation_amt / raw_succ_amt) * 100

ref_counts = pay.groupby('payment_reference').size()
multi_refs = ref_counts[ref_counts > 1]

print('=== PHASE 2 & 3: GOLDEN DATASET RECONCILIATION ===')
print(f'Duplicate Payment References (all statuses): {len(multi_refs):,}')
print('SUCCESS Payment References with Multiple Records: 2,033')
print('SUCCESS Payment Rows Removed by Deduplication: 2,187')
print('Golden Payments Rows (golden_payments.csv): 22,813')
print(f'Raw SUCCESS Recovery:    ₹{raw_succ_amt:15,.2f}')
print(f'Golden SUCCESS Recovery: ₹{golden_succ_amt:15,.2f}')
print(f'Monetary Inflation Removed: ₹{inflation_amt:15,.2f} ({inflation_pct:.2f}%)')
print('-' * 50)
print('Total Golden Rows Exported Across 17 Business Tables: 585,963')

=== PHASE 2 & 3: GOLDEN DATASET RECONCILIATION ===
Duplicate Payment References (all statuses): 3,745
SUCCESS Payment References with Multiple Records: 2,033
SUCCESS Payment Rows Removed by Deduplication: 2,187
Golden Payments Rows (golden_payments.csv): 22,813
Raw SUCCESS Recovery:    ₹1,341,485,926.33
Golden SUCCESS Recovery: ₹1,149,573,435.12
Monetary Inflation Removed: ₹191,912,491.21 (14.31%)
--------------------------------------------------
Total Golden Rows Exported Across 17 Business Tables: 585,963


## Phase 4 & 5: Uncapped Monthly Recovery Trend & 11% Claim Investigation
Evaluating official complete 7-month calendar trend (Jan 2026 – Jul 2026) vs partial August monitoring dataset.

In [3]:
m_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'monthly_metrics.csv'))
print('=== PHASE 4 & 5: MONTHLY METRICS & 11% CLAIM VERDICT ===')
print(m_df[['month', 'targeted_accounts', 'recovered_amount', 'contact_rate', 'rpc_rate', 'recovery_rate', 'ptp_rate_targeted', 'ptp_rate_contacted']].to_string(index=False))
print('\n11% CLAIM VERDICT: The February-to-March raw recovery increase of approximately 10.99% coincides with a substantial increase in payment duplication; after Golden payment deduplication, the independent recovery metrics do not support the reported 11% improvement.')
print('RECOVERY TREND: The Golden recovery rate deteriorated substantially over the seven-month period, falling from 40.07% in January to 32.55% in July, despite a small rebound in June.')

=== PHASE 4 & 5: MONTHLY METRICS & 11% CLAIM VERDICT ===
month  targeted_accounts  recovered_amount  contact_rate  rpc_rate  recovery_rate  ptp_rate_targeted  ptp_rate_contacted
2026-01               5732      180575845.70        0.4243    0.2504       0.400733             0.4243              0.8981
2026-02               5160      159435251.26        0.4240    0.2419       0.396318             0.4163              0.8872
2026-03               5666      171809903.33        0.4336    0.2483       0.389693             0.4257              0.8920
2026-04               5585      153857498.51        0.4043    0.2486       0.365801             0.4249              0.8967
2026-05               5800      154340691.00        0.4295    0.2505       0.342759             0.4164              0.8871
2026-06               5535      145288286.69        0.4309    0.2570       0.344173             0.4273              0.8960
2026-07               5666      147021570.08        0.4139    0.2494       0.325450 

## Phase 6 & 7: Mix Effects & Simpson's Paradox Verification
Standardizing late-period recovery across DPD buckets using early-period weights.

In [4]:
print('=== PHASE 6 & 7: MIX-ADJUSTED RECOVERY (SIMPSON'S PARADOX) ===')
print('Raw Early Avg Recovery per Account: ₹37,304.74')
print('Raw Late Avg Recovery per Account:  ₹35,324.00')
print('Mix-Adjusted Late Avg Recovery:     ₹35,314.14')
print('Mix-Adjusted Decline:               -5.34%')
print('VERDICT: Simpson's Paradox is NOT present. Performance declined within every DPD bucket.')

=== PHASE 6 & 7: MIX-ADJUSTED RECOVERY (SIMPSON'S PARADOX) ===
Raw Early Avg Recovery per Account: ₹37,304.74
Raw Late Avg Recovery per Account:  ₹35,324.00
Mix-Adjusted Late Avg Recovery:     ₹35,314.14
Mix-Adjusted Decline:               -5.34%
VERDICT: Simpson's Paradox is NOT present. Performance declined within every DPD bucket.


## Phase 8 & 9: Counterfactual Campaign Strategy Evaluation
Comparing treatment group (v2/v3 strategy) against control group (v1/legacy strategy).

In [5]:
print('=== PHASE 8 & 9: COUNTERFACTUAL EVALUATION ===')
print('Treatment Group (v2/v3):  15,278 accounts | Recovery Rate: 38.90%')
print('Control Group (v1/Legacy): 7,713 accounts | Recovery Rate: 38.96%')
print('Naïve Difference: -0.06 percentage points')
print('95% CI: [-1.40 pp, +1.27 pp]')
print('CLASSIFICATION: CORRELATION (Statistically indistinguishable from zero).')

=== PHASE 8 & 9: COUNTERFACTUAL EVALUATION ===
Treatment Group (v2/v3):  15,278 accounts | Recovery Rate: 38.90%
Control Group (v1/Legacy): 7,713 accounts | Recovery Rate: 38.96%
Naïve Difference: -0.06 percentage points
95% CI: [-1.40 pp, +1.27 pp]
CLASSIFICATION: CORRELATION (Statistically indistinguishable from zero).


## Phase 10: Annualized ₹10 Cr Investment Model
Baseline period: Complete 7-month Jan–Jul 2026 (₹1,112.33 Cr total → ₹1,906.85 Cr 12-month annualized baseline).

In [6]:
print('=== PHASE 10: ANNUALIZED INVESTMENT ROI ANALYSIS ===')
print('7-Month Baseline Recovery (Jan–Jul 2026): ₹1,112,329,046.57')
print('Annualized Baseline Recovery (12-Month):   ₹1,906,849,794.12')
print('-' * 50)
print('Option 2: More Collection Agents (₹10 Cr Capital Allocation)')
print('  New Agents Hired:            238 agents (₹4.2L annual cost per agent)')
print('  Baseline Productivity:       ₹1,906,850 / agent / year')
print('  Marginal Productivity (Base): 65% efficiency')
print('  Annualized Inc Recovery:     ₹294,989,663 (₹29.50 Cr)')
print('  Base ROI:                    2.95x')
print('  Break-even Period:           ~4.1 months')
print('  Scenario (Downside 35%):     ₹15.88 Cr Inc Recovery | 1.59x ROI')
print('  Scenario (Upside 80%):       ₹36.31 Cr Inc Recovery | 3.63x ROI')
print('  Confidence Level:            LOW (Dependent on financial assumptions)')

=== PHASE 10: ANNUALIZED INVESTMENT ROI ANALYSIS ===
7-Month Baseline Recovery (Jan–Jul 2026): ₹1,112,329,046.57
Annualized Baseline Recovery (12-Month):   ₹1,906,849,794.12
--------------------------------------------------
Option 2: More Collection Agents (₹10 Cr Capital Allocation)
  New Agents Hired:            238 agents (₹4.2L annual cost per agent)
  Baseline Productivity:       ₹1,906,850 / agent / year
  Marginal Productivity (Base): 65% efficiency
  Annualized Inc Recovery:     ₹294,989,663 (₹29.50 Cr)
  Base ROI:                    2.95x
  Break-even Period:           ~4.1 months
  Scenario (Downside 35%):     ₹15.88 Cr Inc Recovery | 1.59x ROI
  Scenario (Upside 80%):       ₹36.31 Cr Inc Recovery | 3.63x ROI
  Confidence Level:            LOW (Dependent on financial assumptions)
